### Requirements

```bash
pip install pycocotools datasets captum
```

### Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

from datasets import load_dataset

import random
import json
from PIL import Image
import pandas as pd
from tqdm import tqdm
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torchvision

# from utils.xai_methods import GradCAMHeatmap, LRP
from utils.utils import SquareCropAndResize, imagenet_transform, inverse_transform, target_transform
from utils.utils import evaluate_xai_methods


def choose_device() -> str:
    if torch.cuda.is_available():
        return "cuda:0"
    if hasattr(torch.backends, "mps"):
        if torch.backends.mps.is_available():
            return "mps"
    return "cpu"

device = torch.device(choose_device())
print(torch.cuda.get_device_name(device))

# fix seed for reproducibility
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed) # multi-GPU



NVIDIA RTX 6000 Ada Generation


In [2]:
from utils.metrics import (
    IoU,
    Precision,
    Recall,
    F1Score,
    DiceCoefficient,
    BoundaryIoU,
    AverageSurfaceDistance,
    AverageSymmetricSurfaceDistance,
    HausdorffDistance,
    NormalizedSurfaceDistance
)

dilation_radius = 3
threshold = 0.3 # only for heatmaps


metrics = {
    'iou': IoU(threshold=threshold),
    'precision': Precision(threshold=threshold),
    'recall': Recall(threshold=threshold),
    'f1': F1Score(threshold=threshold),
    'dice': DiceCoefficient(threshold=threshold),
    'biou': BoundaryIoU(dilation_radius=dilation_radius, threshold=threshold),
    'asd': AverageSurfaceDistance(threshold=threshold),
    'assd': AverageSymmetricSurfaceDistance(dilation_radius=dilation_radius, threshold=threshold),
    'hd': HausdorffDistance(threshold=threshold),
    'nsd': NormalizedSurfaceDistance(threshold=threshold),
}


In [3]:
# Load labels from json file
with open("imagenet_class_index.json") as f:
    imagenet_labels = list(json.load(f).values())


### Load models

In [4]:
resnet = torchvision.models.resnet50(weights="ResNet50_Weights.IMAGENET1K_V2")
resnet.eval().to(device);

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/jovyan/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 162MB/s]


In [5]:
from utils.xai_methods import LRP, GradCAMHeatmap

from zennit.composites import (
    EpsilonPlus,
    EpsilonPlusFlat,
    ExcitationBackprop,
    EpsilonGammaBox,
)
from zennit.canonizers import SequentialMergeBatchNorm
# from zennit.torchvision import ResNetCanonizer




# for visualization
# without interpolation for vector plot (resulting in a heatmap with 7x7)
xai_methods = (
    (
        "GradCAM",
        GradCAMHeatmap(resnet, resnet.layer4),
    ),
    (
        "EP",
        LRP(
            model=resnet,
            composite=EpsilonPlus(canonizers=[SequentialMergeBatchNorm()]),
            LRP_target_layer=None,
        ),
    ),    
    (
        "EP 1.0.conv2",
        LRP(
            model=resnet,
            composite=EpsilonPlus(canonizers=[SequentialMergeBatchNorm()]),
            LRP_target_layer="layer1.0.conv2",
        ),
    ),
    (
        "EP 2.0.conv2",
        LRP(
            resnet,
            composite=EpsilonPlus(canonizers=[SequentialMergeBatchNorm()]),
            LRP_target_layer="layer2.0.conv2",
        ),
    ),
    (
        "EP 3.0.conv2",
        LRP(
            resnet,
            composite=EpsilonPlus(canonizers=[SequentialMergeBatchNorm()]),
            LRP_target_layer="layer3.0.conv2",
        ),
    ),
    (
        "GB",
        LRP(
            resnet,
            composite=EpsilonGammaBox(
                high=3, low=-3, canonizers=[SequentialMergeBatchNorm()]
            ),
            LRP_target_layer=None,
        ),
    ),

    (
        "GB 1.0.conv2",
        LRP(
            resnet,
            composite=EpsilonGammaBox(
                high=3, low=-3, canonizers=[SequentialMergeBatchNorm()]
            ),
            LRP_target_layer="layer1.0.conv2",
        ),
    ),
    (
        "GB 2.0.conv2",
        LRP(
            resnet,
            composite=EpsilonGammaBox(
                high=3, low=-3, canonizers=[SequentialMergeBatchNorm()]
            ),
            LRP_target_layer="layer2.0.conv2",
        ),
    ),
    (
        "GB 3.0.conv2",
        LRP(
            resnet,
            composite=EpsilonGammaBox(
                high=3, low=-3, canonizers=[SequentialMergeBatchNorm()]
            ),
            LRP_target_layer="layer3.0.conv2",
        ),
    ),
)

## COCO

Download the [dataset](https://cocodataset.org/#download) ([train2017.zip](http://images.cocodataset.org/zips/train2017.zip), [val2017.zip](http://images.cocodataset.org/zips/val2017.zip) and [annotations_trainval2017.zip](http://images.cocodataset.org/annotations/annotations_trainval2017.zip))

In [6]:
from torchvision.datasets import CocoDetection

coco_ds = CocoDetection(root="../data/coco/train2017", annFile="../data/coco/annotations_trainval2017/annotations/instances_train2017.json", transform=imagenet_transform, target_transform=None)
coco_ds_val = CocoDetection(root="../data/coco/val2017", annFile="../data/coco/annotations_trainval2017/annotations/instances_val2017.json", transform=imagenet_transform, target_transform=None)


ModuleNotFoundError: No module named 'pycocotools'

Converting COCO dataset to pandas dataframe takes time, so we will do it once and load the dataframe from csv in the future (uncomment the following cell)

In [ ]:
# # convert coco coco_ds to pandas dataframe
# data = []

# for i in range(len(coco_ds)):
#     rec = {
#         "dataset_idx": i,
#         "categories": [],
#         "label_names": [],
#         "supercategories": [],
#         "segmentation_ids": [],
#         "segmentations": [],
#     }
#     for seg in coco_ds[i][1]:
#         # filter out categories not in labels
        
#         rec["segmentations"].append(seg)
#         rec["categories"].append(seg["category_id"])
#         rec["label_names"].append(coco_ds.coco.cats[seg["category_id"]]["name"])
#         rec["supercategories"].append(coco_ds.coco.cats[seg["category_id"]]["supercategory"])
#         rec["segmentation_ids"].append(seg["id"])
#     if len(rec["categories"]) > 0:
#         data.append(rec)

# df_coco = pd.DataFrame(data)
# df_coco.to_csv("coco_ds.csv", index=False)

# df_coco.head()


In [ ]:
df_coco = pd.read_csv("coco_ds.csv")

## strings to list
df_coco[["categories", "label_names", "supercategories", "segmentation_ids", "segmentations"]] = df_coco[["categories", "label_names", "supercategories", "segmentation_ids", "segmentations"]].applymap(eval)

df_coco.head()

In [ ]:
# select labels

labels = {
    # 1: 'person',
    2: 'bicycle',
    3: 'car',
    4: 'motorcycle',
    5: 'airplane',
    6: 'bus',
    7: 'train',
    8: 'truck',
    9: 'boat',
    10: 'traffic light',
    11: 'fire hydrant',
    13: 'stop sign',
    14: 'parking meter',
    15: 'bench',
    16: 'bird',
    17: 'cat',
    18: 'dog',
    19: 'horse',
    20: 'sheep',
    21: 'cow',
    22: 'elephant',
    23: 'bear',
    24: 'zebra',
    25: 'giraffe',
    # 27: 'backpack',
    28: 'umbrella',
    # 31: 'handbag',
    # 32: 'tie',
    # 33: 'suitcase',
    # 34: 'frisbee',
    # 35: 'skis',
    # 36: 'snowboard',
    37: 'sports ball',
    38: 'kite',
    # 39: 'baseball bat',
    # 40: 'baseball glove',
    41: 'skateboard',
    # 42: 'surfboard',
    # 43: 'tennis racket',
    44: 'bottle',
    46: 'wine glass',
    47: 'cup',
    # 48: 'fork',
    # 49: 'knife',
    # 50: 'spoon',
    # 51: 'bowl',
    52: 'banana',
    53: 'apple',
    # 54: 'sandwich',
    55: 'orange',
    56: 'broccoli',
    57: 'carrot',
    58: 'hot dog',
    59: 'pizza',
    # 60: 'donut',
    # 61: 'cake',
    # 62: 'chair',
    # 63: 'couch',
    # 64: 'potted plant',
    # 65: 'bed',
    # 67: 'dining table',
    # 70: 'toilet',
    # 72: 'tv',
    73: 'laptop',
    74: 'mouse',
    75: 'remote',
    76: 'keyboard',
    77: 'cell phone',
    78: 'microwave',
    79: 'oven',
    80: 'toaster',
    # 81: 'sink',
    82: 'refrigerator',
    # 84: 'book',
    85: 'clock',
    86: 'vase',
    87: 'scissors',
    88: 'teddy bear',
    89: 'hair drier',
    # 90: 'toothbrush'
}

# df_single_objects = df_coco[df_coco["label_names"].apply(lambda x: any(label in x for label in labels.values()))]


In [ ]:
# filter with one mask
df_single_objects = df_coco[df_coco["categories"].apply(len) == 1]
# filter with labels
len(df_single_objects)

In [ ]:
# plot number of samples per label

df_single_objects["label_names"].explode().value_counts().plot(
    kind="bar", figsize=(16, 5), title="COCO 2017 - labels distribution",
);


In [ ]:
import matplotlib.colors as colors
from random import randint

color_list = ['black', 'green', 'red', 'blue', 'yellow', 'cyan', 'magenta', 'white', 'gray']

idx = randint(0, len(df_single_objects))
row = df_single_objects.iloc[idx]
img_tensor = coco_ds[int(row["dataset_idx"])][0]


fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

img = inverse_transform(img_tensor).permute(1, 2, 0).numpy()
axes[0].imshow(img)
axes[0].axis("off")

merged_mask = np.zeros(img_tensor.shape[-2:])
for i, seg in enumerate(row["segmentations"]):
    mask = coco_ds.coco.annToMask(seg)
    # crop mask to make it the same shape as the input
    mask = np.array(target_transform(Image.fromarray(mask)))
    merged_mask[mask > 0] = i + 1

axes[1].imshow(merged_mask, cmap=colors.ListedColormap(color_list[:len(np.unique(merged_mask))]))
axes[1].set_title("Segmentation Masks")
axes[1].axis("off")
    


idx, row["label_names"]

In [ ]:
df_sample = df_single_objects.sample(5)

fig, axes = plt.subplots(
    nrows=len(df_sample),
    ncols=len(xai_methods) + 1,
    figsize=((len(xai_methods) + 1) * 2, 2 * len(df_sample)),
)
for ax in axes.flatten():
    ax.axis("off")

for i, row in enumerate(tqdm(df_sample.itertuples(), total=len(df_sample))):
    img_tensor = coco_ds[int(row.dataset_idx)][0]
    img = inverse_transform(img_tensor).permute(1, 2, 0).numpy()

    img_tensor = img_tensor.unsqueeze(0).to(device)

    # there could be multiple masks for one image
    merged_mask = np.zeros(img_tensor.shape[-2:])
    for seg in row.segmentations:
        mask = coco_ds.coco.annToMask(seg)
        mask = np.array(target_transform(Image.fromarray(mask)))
        merged_mask[mask > 0] = 1

    if i == 0:
        axes[i, 0].title.set_text("Original image")

    axes[i, 0].imshow(img)

    with torch.no_grad():
        pred = resnet(img_tensor).argmax()
    pred_label = imagenet_labels[pred]
    pred_label = pred_label.split(",")[0].title()

    # add vertical title left
    true_label = ", ".join(row.label_names)
    axes[i, 0].text(
        -0.1,
        0.5,
        f"True: {true_label}\nPred.: {pred_label}",
        size=12,
        ha="center",
        va="center",
        rotation=90,
        transform=axes[i, 0].transAxes,
    )

    for j, method_info in enumerate(xai_methods):
        name, method = method_info
        heatmap = method(img_tensor, pred)

        # make sure there are no negative values
        heatmap = torch.relu(heatmap)
        # normalize to [0, 1]
        heatmap = heatmap / heatmap.max()
 
        if i == 0:
            axes[i, j + 1].title.set_text(name)
        heatmap = np.flipud(heatmap.cpu().numpy())

        x = np.linspace(0, 224, heatmap.shape[0])
        y = np.linspace(0, 224, heatmap.shape[1])
        X, Y = np.meshgrid(x, y)

        axes[i, j + 1].pcolormesh(X, Y, heatmap, cmap='hot', shading='auto')
        axes[i, j + 1].set_aspect('equal')
        axes[i, j + 1].contour(merged_mask[::-1], levels=[0.5], colors='c', alpha=0.6)

plt.tight_layout()
# remove margins
plt.subplots_adjust(wspace=0, hspace=0.05)

df_sample.loc[:, ["dataset_idx", "label_names"]]

Evaluation example (evaluates samples the from previous cell)

In [ ]:

# df_sample = df_single_objects.sample(5)
metric_results = []

for i, row in enumerate(tqdm(df_sample.itertuples(), total=len(df_sample))):
    img_tensor = coco_ds[int(row.dataset_idx)][0]
    img_tensor = img_tensor.unsqueeze(0).to(device)

    # there could be multiple masks for one image
    merged_mask = np.zeros(img_tensor.shape[-2:])
    for seg in row.segmentations:
        mask = coco_ds.coco.annToMask(seg)
        mask = np.array(target_transform(Image.fromarray(mask)))
        merged_mask[mask > 0] = 1

    with torch.no_grad():
        pred = resnet(img_tensor).argmax()
    pred_label = imagenet_labels[pred]
    pred_label = pred_label.split(",")[0].title()

    # add vertical title left
    true_label = ", ".join(row.label_names)

    for j, method_info in enumerate(xai_methods):
        name, method = method_info
        heatmap = method(img_tensor, pred).cpu()
        # make sure there are no negative values
        heatmap = torch.relu(heatmap)
        # normalize to [0, 1]
        heatmap = heatmap / heatmap.max()

        # upsample heatmap to 224x224 using nearest interpolation with PIL
        heatmap = Image.fromarray(heatmap.numpy())
        heatmap = heatmap.resize((224, 224), Image.NEAREST)
        heatmap = np.array(heatmap)

        metric_results.append(evaluate_xai_methods(merged_mask, heatmap, row.dataset_idx, pred_label, true_label, name, metrics))


pd.DataFrame(metric_results)

# PASCAL Segmentation


In [ ]:
pascal_labels = {
    1: "aeroplane",
    2: "bicycle",
    3: "bird",
    4: "boat",
    5: "bottle",
    6: "bus",
    7: "car",
    8: "cat",
    9: "chair",
    10: "cow",
    11: "diningtable",
    12: "dog",
    13: "horse",
    14: "motorbike",
    15: "person",
    16: "potted plant",
    17: "sheep",
    18: "sofa",
    19: "train",
    20: "tv/monitor",
}


pascal_ds = torchvision.datasets.VOCSegmentation(
    root="../data/VOCdevkit",
    year="2012",
    image_set="val",
    download=False,
    transform=imagenet_transform,
    target_transform=SquareCropAndResize(224, interpolation=Image.NEAREST),
)



### Create pandas dataframe from torch dataset


In [ ]:
data = []

for idx, (_, mask) in enumerate(pascal_ds):
    mask = np.array(mask)
    labels = np.unique(mask)
    labels = labels[(labels != 0) & (labels != 255)]

    data.append(
        {
            "dataset_idx": idx,
            "labels": labels,
            "label_count": len(labels),
            "label_names": [pascal_labels[label] for label in labels],
        }
    )

# Create a pandas dataframe
pascal_df = pd.DataFrame(data)


print(f"Number of images: {len(pascal_df)}")

pascal_df["label_names"].explode().value_counts().plot(
    kind="bar", figsize=(6, 3), title="Pascal VOC 2012 - labels distribution"
)

In [ ]:
pascal_df = pascal_df[pascal_df["label_count"] == 1].copy()

exclude_classes = [
    "person",
    "tv/monitor",
    "sofa",
    "potted plant",
    "diningtable",
    "chair",
    "bottle",
]
pascal_df = pascal_df[
    ~pascal_df["label_names"].apply(lambda x: any(cls in x for cls in exclude_classes))
].copy()

print(f"Number of images: {len(pascal_df)}")

pascal_df["label_names"].explode().value_counts().plot(
    kind="bar", figsize=(6, 3), title="Pascal VOC 2012 - labels distribution"
)

### Visualization

In [ ]:
df_sample = pascal_df.sample(5)

fig, axes = plt.subplots(
    nrows=len(df_sample),
    ncols=len(xai_methods) + 2,
    figsize=((len(xai_methods) + 2) * 2, 2 * len(df_sample)),
)
for ax in axes.flatten():
    ax.axis("off")

for i, row in enumerate(tqdm(df_sample.itertuples(), total=len(df_sample))):
    img_tensor, seg_img = pascal_ds[row.dataset_idx]
    img = inverse_transform(img_tensor).permute(1, 2, 0).numpy()

    img_tensor = img_tensor.unsqueeze(0).to(device)

    seg = np.array(seg_img)
    # remove outline
    seg[seg == 255] = 0

    seg_classes = np.unique(seg)
    seg_classes = seg_classes[seg_classes != 0]
    seg_labels = [pascal_labels[i] for i in seg_classes]
    seg_labels = ", ".join(seg_labels)

    seg = np.array(seg_img)
    mask = np.zeros_like(seg).astype(np.float32)
    mask[seg == 255] = 1
    mask_alpha = plt.cm.gray(mask)
    mask_alpha[..., 3] = mask

    if i == 0:
        axes[i, 0].title.set_text("Original image")

    axes[i, 0].imshow(img, vmin=img.min(), vmax=img.max())

    with torch.no_grad():
        pred = resnet(img_tensor).argmax()
    pred_label = imagenet_labels[pred]
    pred_label = pred_label.split(",")[0].title()

    # add vertical title left
    axes[i, 0].text(
        -0.1,
        0.5,
        f"True: {seg_labels}\nPred.: {pred_label}",
        size=12,
        ha="center",
        va="center",
        rotation=90,
        transform=axes[i, 0].transAxes,
    )

    for j, method_info in enumerate(xai_methods):
        name, method = method_info
        heatmap = method(img_tensor, pred)
        # make sure there are no negative values
        heatmap = torch.relu(heatmap)
        # normalize to [0, 1]
        heatmap = heatmap / heatmap.max()

        if i == 0:
            axes[i, j + 1].title.set_text(name)
        heatmap = np.flipud(heatmap.cpu().numpy())

        x = np.linspace(0, 224, heatmap.shape[0])
        y = np.linspace(0, 224, heatmap.shape[1])
        X, Y = np.meshgrid(x, y)

        axes[i, j + 1].pcolormesh(X, Y, heatmap, cmap='hot', shading='auto')
        axes[i, j + 1].set_aspect('equal')
        axes[i, j + 1].contour(mask[::-1], levels=[0.5], colors='c', alpha=0.6)

plt.tight_layout()
plt.subplots_adjust(wspace=0, hspace=0.05)

df_sample.loc[:, ["dataset_idx", "label_names"]]

### Evaluation


In [ ]:

# df_sample = df_single_objects.sample(5)
metric_results = []

for i, row in enumerate(tqdm(df_sample.itertuples(), total=len(df_sample))):
    img_tensor = pascal_ds[int(row.dataset_idx)][0]
    img_tensor = img_tensor.unsqueeze(0).to(device)

    seg = np.array(seg_img)
    seg[seg == 255] = 1
    mask = np.zeros_like(seg).astype(np.float32)
    mask[seg == 1] = 1


    with torch.no_grad():
        pred = resnet(img_tensor).argmax()
    pred_label = imagenet_labels[pred]
    pred_label = pred_label.split(",")[0].title()
    true_label = ", ".join(row.label_names)

    for j, method_info in enumerate(xai_methods):
        name, method = method_info
        heatmap = method(img_tensor, pred).cpu()
        # make sure there are no negative values
        heatmap = torch.relu(heatmap)
        # normalize to [0, 1]
        heatmap = heatmap / heatmap.max()

        # upsample heatmap to 224x224 using nearest interpolation with PIL
        heatmap = Image.fromarray(heatmap.numpy())
        heatmap = heatmap.resize((224, 224), Image.NEAREST)
        heatmap = np.array(heatmap)

        metric_results.append(evaluate_xai_methods(mask, heatmap, row.dataset_idx, pred_label, true_label, name, metrics))


pd.DataFrame(metric_results)

# ImageNet-1k + ImageNet-S

You may need to accept the huggingface terms and conditions to download the [imagenet-1k](https://huggingface.co/datasets/imagenet-1k) dataset. Also, make sure you have run `huggingface-cli login`.

In [ ]:
imgent_ds = load_dataset("imagenet-1k", streaming=False, split="validation")


### Convert huggingface dataset to dataframe

In [ ]:
rows = []
for i, (img, l) in enumerate(zip(imgent_ds._data["image"], imgent_ds._data["label"])):
    l = l.as_py()
    path = img["path"].as_py()
    row = {}
    row["category"] = path.split("/")[-1].split("_")[-1].split(".")[0]
    row["filename"] = path.split("/")[-1].split("_n")[0]
    row["label_names"] = imagenet_labels[l]
    row["label"] = l
    row["dataset_idx"] = i
    rows.append(row)

# set dtype int instead of numpy.int64
imgent_df = pd.DataFrame(rows)
del rows
imgent_df.head()

### Map ImageNet-1k to ImageNet-S

Before mapping the ImageNet-1k to [ImageNet-S](https://github.com/LUSSeg/ImageNet-S), you need to download [ImageNetS919](https://github.com/LUSSeg/ImageNet-S/releases/download/ImageNet-S/ImageNetS919-5f7f58ae1003d21da9409a8576bf7680.zip). Make sure you have the right path to the ImageNetS919 folder.

In [ ]:
import glob

file_paths = glob.glob("../data/ImageNetS919/validation-segmentation/*/*")
imgnet_s_path = [path.split("/")[-1].split(".")[0] for path in file_paths]
print(len(file_paths), len(imgnet_s_path))
imgnet_s_path[:5]

In [ ]:
# match names and dataframe
imgnet_s_df = imgent_df[imgent_df["filename"].isin(imgnet_s_path)].copy()
# match file_paths and df_seg['filename']
imgnet_s_df["file_path"] = imgnet_s_df["filename"].apply(
    lambda x: file_paths[imgnet_s_path.index(x)]
)

print(len(imgnet_s_df))
imgnet_s_df[['filename', 'file_path']].head()

Extract labels from segmentation masks (see [here](https://github.com/LUSSeg/ImageNet-S?tab=readme-ov-file#qa) for more details).

In [ ]:
def get_seg_id(path):
    segmentation = Image.open(path)  # RGB
    segmentation = np.array(segmentation, dtype=np.uint16)
    seg = segmentation[:, :, 1] * 256 + segmentation[:, :, 0]  # R+G*256
    seg = np.unique(seg)
    seg = seg[(seg != 0) & (seg != 1000)]
    return seg


imgnet_s_df["segmentation_id"] = imgnet_s_df["file_path"].apply(lambda x: get_seg_id(x))
imgnet_s_df.head()

In [ ]:
imgnet_s_df = imgnet_s_df[imgnet_s_df["segmentation_id"].apply(lambda x: len(x)) == 1].copy()
print(len(imgnet_s_df))
imgnet_s_df.head()

In [ ]:
# plot number of samples per label
imgnet_s_df["label_names"].explode().value_counts().plot(
    kind="bar", figsize=(200, 5), title="ImageNet-S - labels distribution"
);

### Saliency map visualization

In [ ]:
df_sample = imgnet_s_df.sample(5)

fig, axes = plt.subplots(
    nrows=len(df_sample),
    ncols=len(xai_methods) + 1,
    figsize=((len(xai_methods) + 1) * 2, 2 * len(df_sample)),
)
for ax in axes.flatten():
    ax.axis("off")

for i, row in enumerate(tqdm(df_sample.itertuples(), total=len(df_sample))):
    img = imgent_ds[int(row.dataset_idx)]['image'].convert("RGB")
    img_tensor = imagenet_transform(img).unsqueeze(0).to(device)


    seg_img = Image.open(row.file_path)
    seg_img = SquareCropAndResize(224)(seg_img)
    seg_img = np.array(seg_img)

    mask = np.array(seg_img[:, :, 0]).astype(np.float32)

    mask[mask != 0] = 1

    if i == 0:
        axes[i, 0].title.set_text("Original image")

    axes[i, 0].imshow(img)

    with torch.no_grad():
        pred = resnet(img_tensor).argmax()
    pred_label = imagenet_labels[pred]
    pred_label = pred_label.split(",")[0].title()
                                     

    # add vertical title left
    true_label = imagenet_labels[row.label]
    true_label = true_label.split(",")[0]
    axes[i, 0].text(
        -0.1,
        0.5,
        f"True: {true_label}\nPred.: {pred_label}",
        size=12,
        ha="center",
        va="center",
        rotation=90,
        transform=axes[i, 0].transAxes,
    )

    for j, method_info in enumerate(xai_methods):
        name, method = method_info
        heatmap = method(img_tensor, pred)
        # make sure there are no negative values
        heatmap = torch.relu(heatmap)
        # normalize to [0, 1]
        heatmap = heatmap / heatmap.max()

        if i == 0:
            axes[i, j + 1].title.set_text(name)
        heatmap = np.flipud(heatmap.cpu().numpy())

        x = np.linspace(0, 224, heatmap.shape[0])
        y = np.linspace(0, 224, heatmap.shape[1])
        X, Y = np.meshgrid(x, y)

        axes[i, j + 1].pcolormesh(X, Y, heatmap, cmap='hot', shading='auto')
        axes[i, j + 1].set_aspect('equal')
        axes[i, j + 1].contour(mask[::-1], levels=[0.5], colors='c', alpha=0.6)

plt.tight_layout()
# remove margins
plt.subplots_adjust(wspace=0, hspace=0.05)

### Evaluation

In [ ]:
metric_results = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    ds_idx = int(row["dataset_idx"])

    img = imgent_ds[int(row.dataset_idx)]['image'].convert("RGB")
    img_tensor = imagenet_transform(img).unsqueeze(0).to(device)

    seg_img = Image.open(row.file_path)
    seg_img = SquareCropAndResize(224)(seg_img)
    seg_img = np.array(seg_img)

    mask = np.array(seg_img[:, :, 0]).astype(np.float32)

    mask[mask != 0] = 1


    pred = resnet(img_tensor).argmax()
    pred_label = imagenet_labels[pred]
    true_label = row.label_names

    for j, method_info in enumerate(xai_methods):
        name, method = method_info
        heatmap = method(img_tensor, pred).cpu()
        # make sure there are no negative values
        heatmap = torch.relu(heatmap)
        # normalize to [0, 1]
        heatmap = heatmap / heatmap.max()

        # upsample heatmap to 224x224 using nearest interpolation with PIL
        heatmap = Image.fromarray(heatmap.numpy())
        heatmap = heatmap.resize((224, 224), Image.NEAREST)
        heatmap = np.array(heatmap)

        metric_results.append(evaluate_xai_methods(mask, heatmap, row.dataset_idx, pred_label, true_label, name, metrics))


pd.DataFrame(metric_results)